# TV3: Bước 4 - Fact-aware Evidence Reranking & Đánh giá NLI End-to-End
## Module 9 & Pipeline Validation: Reranking → NLI Inputs → FastText/BiLSTM Inference

---

### Điểm quan trọng về Kiến trúc Đồ án:
1. **IR & IE không train NLI**: IR/IE chịu trách nhiệm sàng lọc ra **Evidence mới** có độ tin cậy và sự thật cao nhất.
2. **Tạo dữ liệu đầu vào chuẩn hóa NLI**: `(Claim, Evidence, Label)` xuất thành file CSV.
3. **Model-specific Preprocessing & Inference**:
   - Chạy mô hình NLI FastText + BiLSTM đã huấn luyện (`models/best_fasttext_bilstm.pt`) trên 3 thí nghiệm đối chứng:
     - **Thí nghiệm A (Gold Evidence):** Claim + Bằng chứng chuẩn (Upper-bound lý tưởng).
     - **Thí nghiệm B (BM25 Evidence):** Claim + Bằng chứng do BM25 Top 1 tìm thấy.
     - **Thí nghiệm C (BM25 + IE Reranking):** Claim + Bằng chứng do Fact-aware Reranker chọn lọc.
   - Chứng minh định lượng mức độ đóng góp của Fact-aware Reranking đối với bài toán Fact-checking!

In [1]:
import json
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
import py_vncorenlp

PROJECT_ROOT = Path("../..").resolve()
INPUT_PATH = OUTPUT_DIR / "evidence_features.csv"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/retrieval"
FT_DIR = PROJECT_ROOT / "data/processed/fasttext_bilstm"

df_feats = pd.read_csv(INPUT_PATH)

# Chuẩn hóa BM25 theo từng claim
df_feats["bm25_max"] = df_feats.groupby("claim_id")["bm25_score"].transform("max")
df_feats["bm25_norm"] = df_feats["bm25_score"] / (df_feats["bm25_max"] + 1e-6)

# Điểm số kết hợp đa tiêu chí: 0.70 BM25 + 0.30 Fact Score
ALPHA = 0.70
BETA = 0.30
df_feats["rerank_score"] = round(ALPHA * df_feats["bm25_norm"] + BETA * df_feats["fact_score"], 4)

df_feats["bm25_rank"] = df_feats["rank"]
df_feats["rerank_rank"] = df_feats.sort_values(["claim_id", "rerank_score"], ascending=[True, False]).groupby("claim_id").cumcount() + 1

output_rerank_path = OUTPUT_DIR / "reranked_evidence.csv"
df_feats.to_csv(output_rerank_path, index=False)
print(f"✓ Đã lưu bảng xếp hạng lại tại: {output_rerank_path.name}")

✓ Đã lưu bảng xếp hạng lại tại: reranked_evidence.csv


In [2]:
# Đánh giá Recall@1 và MRR trước và sau Reranking
eval_df = df_feats[df_feats["label"] != 2].copy()
n_claims = eval_df["claim_id"].nunique()

def eval_ranks(d, rank_col):
    r1 = d[(d[rank_col] == 1) & (d["is_gold"] == True)]["claim_id"].nunique() / n_claims
    r3 = d[(d[rank_col] <= 3) & (d["is_gold"] == True)]["claim_id"].nunique() / n_claims
    r5 = d[(d[rank_col] <= 5) & (d["is_gold"] == True)]["claim_id"].nunique() / n_claims
    g = d[d["is_gold"] == True]
    mrr = (1.0 / g.groupby("claim_id")[rank_col].min()).sum() / n_claims
    return r1, r3, r5, mrr

bm25_r1, bm25_r3, bm25_r5, bm25_mrr = eval_ranks(eval_df, "bm25_rank")
rerank_r1, rerank_r3, rerank_r5, rerank_mrr = eval_ranks(eval_df, "rerank_rank")

promoted = eval_df[(eval_df["bm25_rank"] > 1) & (eval_df["rerank_rank"] == 1) & (eval_df["is_gold"] == True)]

print("=" * 70)
print("SO SÁNH HIỆU NĂNG RETRIEVAL TRƯỚC VÀ SAU FACT RERANKING:")
print(f"• Recall@1: BM25 = {bm25_r1*100:.2f}%  -->  Fact Reranker = {rerank_r1*100:.2f}% (Delta: {(rerank_r1-bm25_r1)*100:+.2f}%)")
print(f"• MRR:      BM25 = {bm25_mrr:.4f}  -->  Fact Reranker = {rerank_mrr:.4f} (Delta: {rerank_mrr-bm25_mrr:+.4f})")
print(f"• Số câu Bằng chứng Vàng được thăng hạng lên Top-1: {len(promoted)} câu!")
print("=" * 70)

SO SÁNH HIỆU NĂNG RETRIEVAL TRƯỚC VÀ SAU FACT RERANKING:
• Recall@1: BM25 = 89.80%  -->  Fact Reranker = 90.00% (Delta: +0.20%)
• MRR:      BM25 = 0.9297  -->  Fact Reranker = 0.9299 (Delta: +0.0002)
• Số câu Bằng chứng Vàng được thăng hạng lên Top-1: 19 câu!


In [3]:
# Tạo các dataset đầu vào cho NLI
dev_cleaned = pd.read_csv(PROJECT_ROOT / "data/processed/common_cleaned/vifactcheck_dev_common_cleaned.csv")

# 1. Dataset A: Gold Evidence
gold_nli = pd.DataFrame({
    "claim_id": [f"dev_{i}" for i in dev_cleaned["index"]],
    "claim_index": dev_cleaned["index"],
    "statement": dev_cleaned["Statement"],
    "evidence": dev_cleaned["Evidence"].fillna(""),
    "label": dev_cleaned["labels"]
})
gold_nli.to_csv(OUTPUT_DIR / "nli_input_gold.csv", index=False)

# 2. Dataset B: BM25 Top 1 Evidence
bm25_map = df_feats[df_feats["bm25_rank"] == 1].set_index("claim_index")["retrieved_evidence"].to_dict()
bm25_nli = pd.DataFrame({
    "claim_id": [f"dev_{i}" for i in dev_cleaned["index"]],
    "claim_index": dev_cleaned["index"],
    "statement": dev_cleaned["Statement"],
    "evidence": dev_cleaned["index"].map(bm25_map).fillna(""),
    "label": dev_cleaned["labels"]
})
bm25_nli.to_csv(OUTPUT_DIR / "nli_input_bm25.csv", index=False)

# 3. Dataset C: Fact Reranked Top 1 Evidence
rerank_map = df_feats[df_feats["rerank_rank"] == 1].set_index("claim_index")["retrieved_evidence"].to_dict()
rerank_nli = pd.DataFrame({
    "claim_id": [f"dev_{i}" for i in dev_cleaned["index"]],
    "claim_index": dev_cleaned["index"],
    "statement": dev_cleaned["Statement"],
    "evidence": dev_cleaned["index"].map(rerank_map).fillna(""),
    "label": dev_cleaned["labels"]
})
rerank_nli.to_csv(OUTPUT_DIR / "nli_input_reranked.csv", index=False)

print(f"✓ Đã xuất 3 file NLI Input tại {OUTPUT_DIR}:")
print("  - nli_input_gold.csv")
print("  - nli_input_bm25.csv")
print("  - nli_input_reranked.csv")

✓ Đã xuất 3 file NLI Input tại /Users/mivu/Documents/Artificial Intelligence/CS221-NLP/Project-NLP-Fact-Checking/notebooks/TV3_IR_IE_Reranking/outputs:
  - nli_input_gold.csv
  - nli_input_bm25.csv
  - nli_input_reranked.csv


In [4]:
# Nạp Model NLI FastText + BiLSTM đã huấn luyện để kiểm thử
with open(FT_DIR / "config.json") as f:
    config = json.load(f)
with open(FT_DIR / "vocab.json") as f:
    vocab = json.load(f)
emb_mat = np.load(FT_DIR / "embedding_matrix.npy")

class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(hidden_dim, 64), nn.Tanh(), nn.Linear(64, 1))
    def forward(self, x, mask):
        scores = self.attn(x).squeeze(-1).masked_fill(~mask, -1e9)
        return (x * torch.softmax(scores, dim=-1).unsqueeze(-1)).sum(dim=1)

class AttnDualEncoderBiLSTM(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, num_classes=3, dropout_rate=0.3, freeze_embedding=True):
        super().__init__()
        weights = torch.tensor(embedding_matrix, dtype=torch.float32)
        self.embedding = nn.Embedding.from_pretrained(weights, freeze=freeze_embedding, padding_idx=0)
        self.lstm = nn.LSTM(embedding_matrix.shape[1], hidden_dim, num_layers=1, bidirectional=True, batch_first=True)
        d = hidden_dim * 2
        self.pool = AttentionPooling(d)
        self.classifier = nn.Sequential(
            nn.Linear(d * 4 + 1, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout_rate), nn.Linear(64, num_classes)
        )
    def forward(self, c, e):
        c_out, _ = self.lstm(self.embedding(c))
        e_out, _ = self.lstm(self.embedding(e))
        u = self.pool(c_out, c != 0)
        v = self.pool(e_out, e != 0)
        cos_sim = (F.normalize(u, dim=-1) * F.normalize(v, dim=-1)).sum(dim=-1, keepdim=True)
        return self.classifier(torch.cat([u, v, torch.abs(u - v), u * v, cos_sim], dim=1))

model = AttnDualEncoderBiLSTM(emb_mat, hidden_dim=128, freeze_embedding=True)
ckpt = torch.load(PROJECT_ROOT / "models/best_fasttext_bilstm.pt", map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# Model-specific preprocessing
conda_prefix = Path(sys.prefix)
os.environ["JAVA_HOME"] = str(conda_prefix)
jvm_path = conda_prefix / "lib" / "server" / "libjvm.dylib"
if jvm_path.exists():
    os.environ["JVM_PATH"] = str(jvm_path)

orig_cwd = Path.cwd()
rdr = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(Path.home() / ".vncorenlp"))
os.chdir(orig_cwd)

def encode_texts(text_list, max_len):
    X = np.zeros((len(text_list), max_len), dtype=np.int32)
    for i, t in enumerate(text_list):
        if not isinstance(t, str) or not t.strip():
            continue
        seg = rdr.word_segment(t)
        tokens = ' '.join(seg).lower().split() if seg else t.lower().split()
        ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens[:max_len]]
        X[i, :len(ids)] = ids
    return X

X_c = np.load(FT_DIR / "X_claim_dev.npy")
y_dev = np.load(FT_DIR / "y_dev.npy")

def evaluate_nli(X_claim, X_evid, y_true):
    ds = TensorDataset(torch.tensor(X_claim, dtype=torch.long), torch.tensor(X_evid, dtype=torch.long), torch.tensor(y_true, dtype=torch.long))
    loader = DataLoader(ds, batch_size=64, shuffle=False)
    preds, trues = [], []
    with torch.no_grad():
        for bc, be, by in loader:
            preds.extend(model(bc, be).argmax(dim=-1).cpu().numpy())
            trues.extend(by.cpu().numpy())
    return accuracy_score(trues, preds), f1_score(trues, preds, average="macro")

# Chạy đánh giá 3 Thí nghiệm
X_e_gold = np.load(FT_DIR / "X_evidence_dev.npy")
acc_a, f1_a = evaluate_nli(X_c, X_e_gold, y_dev)

X_e_bm25 = encode_texts(bm25_nli["evidence"].tolist(), max_len=config["max_len_evidence"])
acc_b, f1_b = evaluate_nli(X_c, X_e_bm25, y_dev)

X_e_rerank = encode_texts(rerank_nli["evidence"].tolist(), max_len=config["max_len_evidence"])
acc_c, f1_c = evaluate_nli(X_c, X_e_rerank, y_dev)

exp_summary = pd.DataFrame([
    {"Thí nghiệm (Experiment)": "A. Gold Evidence (Upper bound)", "Loại Bằng chứng": "Bằng chứng chuẩn gán nhãn tay", "Accuracy": f"{acc_a*100:.2f}%", "Macro-F1": f"{f1_a:.4f}"},
    {"Thí nghiệm (Experiment)": "B. BM25 Evidence", "Loại Bằng chứng": "Bằng chứng Top 1 từ BM25 thuần túy", "Accuracy": f"{acc_b*100:.2f}%", "Macro-F1": f"{f1_b:.4f}"},
    {"Thí nghiệm (Experiment)": "C. Fact-aware Reranked Evidence", "Loại Bằng chứng": "Bằng chứng Top 1 sau khi Rerank đặc trưng Fact", "Accuracy": f"{acc_c*100:.2f}%", "Macro-F1": f"{f1_c:.4f}"}
])

display(exp_summary)
exp_summary.to_csv(OUTPUT_DIR / "nli_comparison_metrics.csv", index=False)

2026-09-07 00:23:02 INFO  WordSegmenter:24 - Loading Word Segmentation model


,Thí nghiệm (Experiment),Loại Bằng chứng,Accuracy,Macro-F1
0,A. Gold Evidence (Upper bound),Bằng chứng chuẩn gán nhãn tay,59.75%,0.6010
1,B. BM25 Evidence,Bằng chứng Top 1 từ BM25 thuần túy,40.80%,0.4062
2,C. Fact-aware Reranked Evidence,Bằng chứng Top 1 sau khi Rerank đặc trưng Fact,40.66%,0.4049
